In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "Which treatments have the best outcomes in Long COVID?"

## Abstract

This analysis examines 6,815 treatment reports from 1,121 unique users on r/covidlonghaulers over one month (March 11 -- April 10, 2026) to identify which treatments the Long COVID community reports as most beneficial. After filtering generic terms, merging duplicate canonicals, and excluding causal-context vaccines, user-level sentiment analysis reveals that micronutrients (magnesium, quercetin, electrolytes, B vitamins) and low dose naltrexone (LDN) consistently achieve the highest positive-outcome rates, while SSRIs and cromolyn sodium are the most polarizing. Treatments are ranked by Wilson-score lower bound to penalize small samples, with confidence intervals and effect sizes reported throughout. Recommendations are tiered by statistical strength.

## 1. Data Exploration

Data covers: **2026-03-11 to 2026-04-10 (~1 month)** from r/covidlonghaulers.

| Metric | Count |
|:---|---:|
| Total users | 2,827 |
| Users with treatment reports | 1,121 |
| Total treatment reports | 6,815 |
| Unique drug names | 1,257 |
| Posts | 17,182 |

**Sentiment distribution across all reports:** 67% positive, 24% negative, 9% mixed, <1% neutral. This community skews positive in its reporting -- people tend to share what works more than what fails, a selection bias we must keep in mind.

In [ ]:
# ── Data exploration: sentiment breakdown & date range ──
sent_q = pd.read_sql('''
    SELECT sentiment, COUNT(*) as n FROM treatment_reports GROUP BY sentiment ORDER BY n DESC
''', conn)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart of sentiment distribution
colors_pie = [COLORS.get('positive', '#2ecc71'), COLORS.get('negative', '#e74c3c'),
              '#f39c12', '#95a5a6']
sent_order = ['positive', 'negative', 'mixed', 'neutral']
sent_vals = [sent_q[sent_q.sentiment == s]['n'].values[0] if s in sent_q.sentiment.values else 0 for s in sent_order]
wedges, texts, autotexts = axes[0].pie(sent_vals, labels=[s.title() for s in sent_order],
                                        colors=colors_pie, autopct='%1.0f%%', startangle=90,
                                        textprops={'fontsize': 11})
axes[0].set_title('Overall Sentiment Distribution\n(All 6,815 Reports)', fontsize=13, fontweight='bold')

# Bar chart of reports over time
time_q = pd.read_sql('''
    SELECT DATE(p.post_date, 'unixepoch') as post_dt, COUNT(*) as n
    FROM treatment_reports tr
    JOIN posts p ON tr.post_id = p.post_id
    GROUP BY post_dt ORDER BY post_dt
''', conn)
time_q['post_dt'] = pd.to_datetime(time_q['post_dt'])
axes[1].bar(time_q['post_dt'], time_q['n'], color='#3498db', alpha=0.7, width=0.8)
axes[1].set_title('Treatment Reports Over Time', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Reports per Day')
axes[1].tick_params(axis='x', rotation=45)

fig.tight_layout()
plt.show()


The left panel shows the overall sentiment skew: two-thirds of reports are positive, reflecting the community's tendency to share successful treatments. The right panel shows reporting is roughly uniform across the month, with no major spikes that would bias time-specific treatments.

**Filtering applied throughout this notebook:**
1. **Generic terms excluded:** "supplements," "medication," "treatment," "therapy," "drug," "vitamin," "antibiotics," "antihistamines" -- these are categories, not actionable treatments
2. **Causal-context vaccines excluded:** COVID vaccines (Pfizer, Moderna, boosters, etc.) show 80-100% negative sentiment because users reference them as *causes* of their condition, not treatments they tried for relief
3. **Duplicate canonicals merged:** famotidine/Pepcid, beta blocker/propranolol, SSRI/fluvoxamine/sertraline, magnesium/magnesium glycinate, h1 antihistamine/cetirizine/fexofenadine/ketotifen, antivirals/paxlovid -- merged at user level to avoid double-counting

## 2. Treatment Ranking by Community Outcome

Before testing specific hypotheses, we need a reliable ranking. Raw positive percentages are misleading when sample sizes vary widely (n=183 for LDN vs n=20 for gabapentin). We rank by **Wilson score lower bound** -- the lower edge of the 95% confidence interval for the positive rate. This penalizes treatments with few reporters, surfacing only those with both high rates *and* adequate evidence.

All analysis is at the **user level**: each person counts once per treatment, based on their average sentiment score across all their reports for that treatment. A user is "positive" if their average score > 0.

In [ ]:
# ── Merge plan for duplicate canonicals ──
import math

MERGE_MAP = {
    'pepcid': 'famotidine',
    'propranolol': 'beta blocker',
    'magnesium glycinate': 'magnesium',
    'h1 antihistamine': 'antihistamines_specific',
    'cetirizine': 'antihistamines_specific',
    'fexofenadine': 'antihistamines_specific',
    'ketotifen': 'antihistamines_specific',
    'fluvoxamine': 'ssri',
    'sertraline': 'ssri',
    'duloxetine': 'ssri',
    'paxlovid': 'antivirals',
    'antidepressants': 'ssri',
}

RENAME_MAP = {
    'antihistamines_specific': 'H1 Antihistamines (combined)',
    'beta blocker': 'Beta Blockers (combined)',
    'famotidine': 'Famotidine / Pepcid',
    'ssri': 'SSRIs / Antidepressants (combined)',
    'antivirals': 'Antivirals / Paxlovid',
    'magnesium': 'Magnesium (all forms)',
}

CAUSAL_EXCLUDE = {
    'vaccine', 'covid vaccine', 'flu vaccine', 'mmr vaccine', 'moderna vaccine',
    'mrna covid-19 vaccine', 'pfizer vaccine', 'vaccine injection', 'pfizer',
    'booster', 'moderna', 'snri',
}

# Build user-level aggregated dataset
raw = pd.read_sql('''
    SELECT tr.user_id, t.canonical_name as drug,
           AVG(CASE tr.sentiment
               WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
               WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_score,
           tr.sentiment
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    GROUP BY tr.user_id, t.canonical_name
''', conn)

# Filter generics and causal
raw = raw[~raw.drug.isin(GENERIC_TERMS | CAUSAL_EXCLUDE)].copy()

# Apply merges
raw['drug_merged'] = raw['drug'].map(lambda d: MERGE_MAP.get(d, d))

# Re-aggregate at user-drug_merged level
user_drug = raw.groupby(['user_id', 'drug_merged']).agg(avg_score=('avg_score', 'mean')).reset_index()
user_drug['is_positive'] = (user_drug.avg_score > 0).astype(int)

# Compute stats per treatment
drug_stats = user_drug.groupby('drug_merged').agg(
    n_users=('user_id', 'nunique'),
    n_positive=('is_positive', 'sum'),
    mean_score=('avg_score', 'mean')
).reset_index()

drug_stats['pos_rate'] = drug_stats.n_positive / drug_stats.n_users
ci = drug_stats.apply(lambda r: wilson_ci(r.n_positive, r.n_users), axis=1)
drug_stats['ci_lo'] = ci.apply(lambda x: x[0])
drug_stats['ci_hi'] = ci.apply(lambda x: x[1])
drug_stats['wilson_lb'] = drug_stats.ci_lo

# Filter to >= 20 users
top = drug_stats[drug_stats.n_users >= 20].sort_values('wilson_lb', ascending=False).reset_index(drop=True)
top['display_name'] = top.drug_merged.map(lambda d: RENAME_MAP.get(d, d.title()))

# Overall baseline
baseline_rate = user_drug.is_positive.mean()


In [ ]:
# ── Forest plot: Wilson-score ranked treatments ──
plot_df = top.head(25).sort_values('wilson_lb', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 10))

y_pos = range(len(plot_df))
colors_forest = ['#2ecc71' if r.pos_rate > baseline_rate else '#e74c3c' if r.pos_rate < 0.5 else '#f39c12'
                 for _, r in plot_df.iterrows()]

# Plot CIs
for i, (_, row) in enumerate(plot_df.iterrows()):
    color = colors_forest[i]
    ax.plot([row.ci_lo, row.ci_hi], [i, i], color=color, linewidth=2, alpha=0.7)
    ax.scatter(row.pos_rate, i, color=color, s=80, zorder=5, edgecolors='white', linewidth=0.5)
    ax.annotate(f"  {row.pos_rate:.0%} (n={row.n_users})",
                xy=(row.ci_hi + 0.01, i), va='center', fontsize=9)

ax.axvline(baseline_rate, color='grey', linestyle='--', alpha=0.6, label=f'Baseline: {baseline_rate:.0%}')
ax.axvline(0.5, color='#e74c3c', linestyle=':', alpha=0.4, label='50% (chance)')
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df.display_name, fontsize=10)
ax.set_xlabel('User-Level Positive Outcome Rate (95% Wilson CI)', fontsize=11)
ax.set_title('Long COVID Treatment Outcomes: Ranked by Wilson Score Lower Bound\n(Minimum 20 user reports)',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0.15, 1.08)

fig.tight_layout()
plt.show()


**How to read this chart:** Each dot shows the percentage of users who reported a positive outcome for that treatment. The horizontal line is the 95% confidence interval -- wider lines mean less certainty. Treatments are ranked by the lower bound of this interval (the worst plausible positive rate), so treatments at the top have both high positive rates *and* enough data to be confident.

**Key findings:**
- **Quercetin** (96%) and **Magnesium** (93%) top the chart with near-universal positive reports, though quercetin's small sample (n=28) means its CI is wider
- **Electrolytes** (88%), **B vitamins** (89%), and **B12** (79%) -- basic nutritional support performs remarkably well
- **Low Dose Naltrexone (LDN)** at 74% positive with n=183 is the most discussed treatment and the strongest evidence base
- **SSRIs** (46%) and **Cromolyn sodium** (35%) fall below chance -- more users report negative outcomes than positive
- **Gabapentin** (45%) also underperforms, notable given its common prescription for neuropathic symptoms

The baseline positive rate across all treatments is 74% -- meaning the average treatment in this community is reported positively. This reflects reporting bias (people share successes more) and should not be interpreted as "74% of treatments work." 

## 3. Statistical Comparisons: What Stands Out?

The forest plot shows a clear spread, but visual differences can be misleading. This section tests whether the top and bottom performers are statistically different from the population baseline, and from each other.

In [ ]:
# ── Binomial tests vs 50% baseline for each treatment ──
results = []
for _, row in top.iterrows():
    bt = binomtest(int(row.n_positive), int(row.n_users), 0.5, alternative='two-sided')
    # Cohen's h effect size
    p1 = row.pos_rate
    p2 = 0.5
    cohens_h = 2 * (math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))
    # NNT vs baseline
    nnt_val = nnt(row.pos_rate, baseline_rate) if row.pos_rate > baseline_rate else None
    results.append({
        'Treatment': row.display_name,
        'n': int(row.n_users),
        'Positive %': f"{row.pos_rate:.0%}",
        'vs 50% p-value': bt.pvalue,
        "Cohen's h": round(cohens_h, 2),
        'Wilson CI': f"[{row.ci_lo:.0%}, {row.ci_hi:.0%}]",
        'NNT vs baseline': f"{nnt_val:.1f}" if nnt_val else chr(8212),
        'pos_rate_raw': row.pos_rate,
    })

results_df = pd.DataFrame(results)

def style_pval(val):
    if val < 0.001:
        return "< 0.001 ***"
    elif val < 0.01:
        return f"{val:.3f} **"
    elif val < 0.05:
        return f"{val:.3f} *"
    else:
        return f"{val:.3f}"

display_df = results_df[['Treatment', 'n', 'Positive %', 'vs 50% p-value', "Cohen's h", 'Wilson CI', 'NNT vs baseline']].copy()
display_df['vs 50% p-value'] = display_df['vs 50% p-value'].apply(style_pval)
display_df.columns = ['Treatment', 'n', 'Positive Rate', 'p vs 50%', "Cohen's h", '95% CI', 'NNT vs Baseline']

display(HTML('<h4>Binomial Test: Each Treatment vs. 50% Chance Baseline</h4>'))
display(HTML(display_df.to_html(index=False, escape=False, classes='table')))
display(HTML("<p style='font-size:0.9em; color:#666;'>*** p&lt;0.001, ** p&lt;0.01, * p&lt;0.05. Cohen's h: small=0.2, medium=0.5, large=0.8. NNT = number needed to treat vs population baseline (74%).</p>"))


**Interpretation:** Most treatments significantly exceed the 50% chance baseline -- unsurprising given the community's positive reporting bias. The more interesting comparisons are treatments that *fail* to reach significance:

- **SSRIs** (p=0.764) and **gabapentin** (p=0.824) cannot be distinguished from a coin flip in this data
- **Cromolyn sodium** is the only treatment significantly *below* 50% (35%, p=0.035) -- a notable finding given it is an FDA-approved mast cell stabilizer often recommended for MCAS (mast cell activation syndrome), a common Long COVID comorbidity

The NNT column shows practical impact vs the population baseline of 74%. Quercetin's NNT of 4.5 means roughly 1 in 5 additional patients who try quercetin will report benefit beyond the baseline rate. Magnesium's NNT of 5.3 is similarly strong.

In [ ]:
# ── Fisher's exact: Top 5 vs Bottom 5 head-to-head ──
top5_names = top.head(5).drug_merged.tolist()
bot5_names = top.tail(5).drug_merged.tolist()

top5_users = user_drug[user_drug.drug_merged.isin(top5_names)]
bot5_users = user_drug[user_drug.drug_merged.isin(bot5_names)]

top5_pos = top5_users.is_positive.sum()
top5_n = len(top5_users)
bot5_pos = bot5_users.is_positive.sum()
bot5_n = len(bot5_users)

table = [[top5_pos, top5_n - top5_pos],
         [bot5_pos, bot5_n - bot5_pos]]
odds, fisher_p = fisher_exact(table)

p1 = top5_pos / top5_n
p2 = bot5_pos / bot5_n
h = 2 * (math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))

top5_display = ', '.join(top.head(5).display_name.tolist())
bot5_display = ', '.join(top.tail(5).display_name.tolist())

display(HTML(f'''
<div style="background: #f8f9fa; padding: 15px; border-radius: 8px; border-left: 4px solid #3498db; margin: 10px 0;">
<h4 style="margin-top:0;">Head-to-Head: Top 5 vs Bottom 5 Treatments</h4>
<table style="font-size:1.05em;">
<tr><td><b>Top 5</b> ({top5_display})</td>
    <td>{p1:.0%} positive (n={top5_n})</td></tr>
<tr><td><b>Bottom 5</b> ({bot5_display})</td>
    <td>{p2:.0%} positive (n={bot5_n})</td></tr>
<tr><td colspan="2"><br/><b>Fisher's exact test:</b> OR = {odds:.2f}, p {"< 0.001" if fisher_p < 0.001 else f"= {fisher_p:.4f}"}</td></tr>
<tr><td colspan="2"><b>Cohen's h:</b> {h:.2f} ({"large" if abs(h) >= 0.8 else "medium" if abs(h) >= 0.5 else "small"} effect)</td></tr>
<tr><td colspan="2"><b>Plain language:</b> Users trying top-performing treatments are roughly {odds:.1f}x as likely to report a positive outcome compared to the bottom performers.</td></tr>
</table>
</div>
'''))


The difference between the best and worst treatments is not subtle. The top tier (micronutrients and targeted immunomodulators) produces positive outcomes at rates far exceeding the bottom tier (SSRIs, gabapentin, steroids). This gap persists even after accounting for sample size differences via Wilson scoring.

## 4. Sentiment Breakdown by Treatment

The previous section looked at binary outcomes (positive vs not). But "not positive" is not a single thing -- a treatment that produces 40% positive and 50% mixed is different from one that produces 40% positive and 50% negative. This diverging bar chart shows the full sentiment breakdown.

In [ ]:
# ── Diverging bar chart: full sentiment breakdown ──
generics_str = ','.join(f"'{g}'" for g in GENERIC_TERMS | CAUSAL_EXCLUDE)
report_sent = pd.read_sql(f'''
    SELECT t.canonical_name as drug, tr.sentiment, COUNT(*) as n
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE t.canonical_name NOT IN ({generics_str})
    GROUP BY t.canonical_name, tr.sentiment
''', conn)

report_sent['drug_merged'] = report_sent.drug.map(lambda d: MERGE_MAP.get(d, d))
report_sent = report_sent.groupby(['drug_merged', 'sentiment']).n.sum().reset_index()

pivot = report_sent.pivot_table(index='drug_merged', columns='sentiment', values='n', fill_value=0)
for col in ['positive', 'negative', 'mixed', 'neutral']:
    if col not in pivot.columns:
        pivot[col] = 0

pivot['total'] = pivot.sum(axis=1)
pivot = pivot[pivot.total >= 25]
pivot['pos_pct'] = pivot.positive / pivot.total * 100
pivot['neg_pct'] = pivot.negative / pivot.total * 100
pivot['mix_pct'] = (pivot.mixed + pivot.neutral) / pivot.total * 100
pivot = pivot.sort_values('pos_pct', ascending=True).tail(20)
pivot['display'] = pivot.index.map(lambda d: RENAME_MAP.get(d, d.title()))

fig, ax = plt.subplots(figsize=(13, 9))
y = range(len(pivot))

# Correct stacking order: mixed innermost, negative outermost
ax.barh(y, -pivot.mix_pct, left=0, color='#95a5a6', height=0.7, label='Mixed / Neutral')
ax.barh(y, -pivot.neg_pct, left=-pivot.mix_pct, color='#e74c3c', height=0.7, label='Negative')
ax.barh(y, pivot.pos_pct, left=0, color='#2ecc71', height=0.7, label='Positive')

ax.set_yticks(y)
ax.set_yticklabels(pivot.display, fontsize=10)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel(chr(8592) + ' Negative / Mixed          Positive ' + chr(8594) + '     (% of reports)', fontsize=11)
ax.set_title('Treatment Sentiment Breakdown\n(Treatments with 25+ reports)', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.0, -0.08), ncol=3, fontsize=10, frameon=False)
ax.set_xlim(-85, 105)

# Percentage labels on positive side
for i, (_, row) in enumerate(pivot.iterrows()):
    if row.pos_pct > 5:
        ax.text(row.pos_pct + 1.5, i, f"{row.pos_pct:.0f}%", va='center', fontsize=9, color='#2ecc71', fontweight='bold')

fig.tight_layout()
plt.show()


**Key pattern:** Treatments at the top (magnesium, electrolytes, B12) have tiny negative bars -- almost nobody reports harm. Treatments at the bottom (SSRIs, cromolyn sodium) have substantial negative and mixed bars, meaning they are *polarizing* rather than universally ineffective. A polarizing treatment is still worth discussing with a doctor if you match the subgroup that benefits; a universally negative treatment is harder to justify.

## 5. Does the Best Treatment Depend on Your Condition?

Long COVID is not one disease. Users with POTS (postural orthostatic tachycardia syndrome), MCAS (mast cell activation syndrome), ME/CFS (myalgic encephalomyelitis / chronic fatigue syndrome), or PEM (post-exertional malaise) may respond differently to the same treatment. This heatmap shows positive outcome rates for top treatments stratified by the user's reported co-occurring condition.

In [ ]:
# ── Heatmap: treatment x condition ──
COMMUNITY_CONDITIONS = {'long covid', 'covid related', 'covid induced', 'post-viral'}

user_conds = pd.read_sql('''
    SELECT DISTINCT user_id, condition_name
    FROM conditions
    WHERE condition_name NOT IN ({})
'''.format(','.join(f"'{c}'" for c in COMMUNITY_CONDITIONS)), conn)

top_conds = user_conds.groupby('condition_name').user_id.nunique().sort_values(ascending=False).head(6).index.tolist()
top_treats = top.head(12).drug_merged.tolist()

rows_ht = []
for cond in top_conds:
    cond_users = set(user_conds[user_conds.condition_name == cond].user_id)
    for treat in top_treats:
        treat_users = user_drug[(user_drug.drug_merged == treat) & (user_drug.user_id.isin(cond_users))]
        n = len(treat_users)
        if n >= 3:
            pos_rate = treat_users.is_positive.mean()
            rows_ht.append({'condition': cond.upper(), 'treatment': RENAME_MAP.get(treat, treat.title()),
                        'pos_rate': pos_rate, 'n': n})

heatmap_df = pd.DataFrame(rows_ht)
if len(heatmap_df) > 0:
    heat_pivot = heatmap_df.pivot_table(index='treatment', columns='condition', values='pos_rate')
    n_pivot = heatmap_df.pivot_table(index='treatment', columns='condition', values='n')

    annot = heat_pivot.copy().astype(object)
    for col in annot.columns:
        for idx in annot.index:
            val = heat_pivot.loc[idx, col] if idx in heat_pivot.index and col in heat_pivot.columns else None
            n_val = n_pivot.loc[idx, col] if idx in n_pivot.index and col in n_pivot.columns else None
            if pd.notna(val) and pd.notna(n_val):
                annot.loc[idx, col] = f"{val:.0%}\n(n={int(n_val)})"
            else:
                annot.loc[idx, col] = ""

    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(heat_pivot, annot=annot.values, fmt='', cmap='RdYlGn', center=0.5,
                vmin=0.2, vmax=1.0, linewidths=0.5, ax=ax, cbar_kws={'label': 'Positive Rate', 'shrink': 0.8},
                annot_kws={'fontsize': 9})
    ax.set_title('Positive Outcome Rate by Treatment and Co-occurring Condition\n(Cells with n < 3 are blank)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

    fig.tight_layout(rect=[0, 0, 0.95, 1])
    plt.show()
else:
    display(HTML('<p><i>Insufficient cross-tabulation data for heatmap.</i></p>'))


**How to read this heatmap:** Green cells indicate high positive outcome rates; red/yellow indicates low. The sample size in each cell (n=X) shows how many users reported on that treatment while also having that condition. Blank cells mean fewer than 3 users overlap -- too few to display.

**Key patterns to note:** Treatments like LDN and magnesium maintain high positive rates across most conditions. But some treatments show condition-specific patterns that merit further investigation -- for example, whether beta blockers perform better among POTS patients (where they have a clear physiological rationale) versus ME/CFS patients (where they may be less relevant).

Note that these cross-tabulations have small cell sizes. Treat condition-specific patterns as hypotheses, not conclusions.

## 6. Counterintuitive Findings Worth Investigating

This section flags results that contradict clinical guidelines, community assumptions, or common sense. These are not conclusions -- they are patterns worth investigating further.

In [ ]:
# ── Counterintuitive findings analysis ──

# 1. Cromolyn sodium
cromo = drug_stats[drug_stats.drug_merged == 'cromolyn sodium']
cromo_rate = cromo.pos_rate.values[0] if len(cromo) > 0 else 0
cromo_n = int(cromo.n_users.values[0]) if len(cromo) > 0 else 0

# 2. SSRIs
ssri_row = drug_stats[drug_stats.drug_merged == 'ssri']
ssri_rate = ssri_row.pos_rate.values[0] if len(ssri_row) > 0 else 0
ssri_n = int(ssri_row.n_users.values[0]) if len(ssri_row) > 0 else 0

# Fluvoxamine isolated
fluv_raw = raw[raw.drug == 'fluvoxamine']
fluv_user = fluv_raw.groupby('user_id').avg_score.mean()
fluv_pos_rate = (fluv_user > 0).mean() if len(fluv_user) > 0 else 0
fluv_n = len(fluv_user)

# 3. Nicotine
nic = drug_stats[drug_stats.drug_merged == 'nicotine']
nic_rate = nic.pos_rate.values[0] if len(nic) > 0 else 0
nic_n = int(nic.n_users.values[0]) if len(nic) > 0 else 0

# 4. Micronutrients vs prescriptions
micro_names = ['magnesium', 'electrolyte', 'b vitamins', 'b12', 'vitamin c', 'vitamin d', 'coq10', 'quercetin', 'omega-3', 'creatine', 'taurine']
rx_names = ['ssri', 'beta blocker', 'famotidine', 'antivirals', 'gabapentin', 'steroids', 'cromolyn sodium', 'guanfacine', 'ivabradine']

micro_users = user_drug[user_drug.drug_merged.isin(micro_names)]
rx_users = user_drug[user_drug.drug_merged.isin(rx_names)]

micro_pos_rate = micro_users.is_positive.mean()
rx_pos_rate = rx_users.is_positive.mean()

table_mr = [[int(micro_users.is_positive.sum()), len(micro_users) - int(micro_users.is_positive.sum())],
            [int(rx_users.is_positive.sum()), len(rx_users) - int(rx_users.is_positive.sum())]]
odds_mr, p_mr = fisher_exact(table_mr)
h_mr = 2 * (math.asin(math.sqrt(micro_pos_rate)) - math.asin(math.sqrt(rx_pos_rate)))

fluv_comment = "which is similarly poor" if fluv_pos_rate < 0.55 else "slightly better but still underwhelming"

display(HTML(f'''
<div style="padding: 15px; border-radius: 8px;">

<h4>Finding 1: Cromolyn Sodium Underperforms Despite Being a Standard MCAS Treatment</h4>
<p>Cromolyn sodium (a mast cell stabilizer) is frequently recommended for MCAS, one of the most common Long COVID comorbidities in this community (75 users report MCAS). Yet it has the <b>lowest</b> positive outcome rate of any treatment with 20+ users: <b>{cromo_rate:.0%}</b> positive (n={cromo_n}). This is significantly below the 50% baseline (binomial p=0.035). Given that MCAS affects 75 users in this dataset and cromolyn is a first-line treatment, this gap between clinical recommendation and community experience deserves investigation.</p>

<h4>Finding 2: SSRIs Are the Most Polarizing Treatment Class</h4>
<p>SSRIs (combined with fluvoxamine, sertraline, duloxetine, and generic antidepressant mentions) show a <b>{ssri_rate:.0%}</b> positive rate (n={ssri_n}). This is essentially a coin flip. Yet fluvoxamine specifically has been studied in clinical trials for Long COVID. When isolated, fluvoxamine alone shows {fluv_pos_rate:.0%} positive (n={fluv_n}), {fluv_comment}. The community's experience with SSRIs is notably worse than the clinical research would suggest.</p>

<h4>Finding 3: Nicotine Patches -- An Unconventional Treatment That Outperforms</h4>
<p>Nicotine (primarily patches) achieves <b>{nic_rate:.0%}</b> positive outcomes from {nic_n} users. For a substance most associated with addiction, this is a surprisingly strong showing. The r/covidlonghaulers community has championed nicotine patches based on the nicotinic acetylcholine receptor hypothesis. This data suggests the community's enthusiasm has some empirical backing in reported outcomes -- though these are self-reports, not clinical measurements.</p>

<h4>Finding 4: Micronutrients Outperform Prescription Medications</h4>
<p>When grouped into categories, over-the-counter micronutrients (magnesium, B vitamins, CoQ10, electrolytes, etc.) achieve <b>{micro_pos_rate:.0%}</b> positive outcomes (n={len(micro_users)}) versus <b>{rx_pos_rate:.0%}</b> for prescription medications (SSRIs, beta blockers, gabapentin, etc., n={len(rx_users)}). Fisher's exact: OR={odds_mr:.2f}, p {"< 0.001" if p_mr < 0.001 else f"= {p_mr:.4f}"}, Cohen's h={h_mr:.2f}. This does not mean supplements are "better" -- it likely reflects that micronutrient deficiencies are common in Long COVID and easier to correct, while prescriptions target more complex pathology. But the magnitude of the gap is worth noting.</p>

</div>
'''))


These findings should not be interpreted as evidence that prescription medications are ineffective or that micronutrients should replace them. The reporting bias in this community -- where people are more likely to share cheap, accessible interventions that worked -- could account for much of the gap. Additionally, people taking prescription medications may have more severe illness, creating confounding by indication.

## 7. What Patients Are Saying *(experimental)*

Every quote below was pulled from r/covidlonghaulers posts in the dataset, selected to illustrate specific treatment outcomes. At least one quote complicates the positive narrative above.

In [ ]:
# ── Pull representative quotes ──
import re

def clean_quote(text, max_len=200):
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    if len(text) > max_len:
        cut = text[:max_len].rfind('.')
        if cut > 80:
            text = text[:cut+1]
        else:
            text = text[:max_len] + '...'
    return text

ldn_pos = pd.read_sql('''
    SELECT p.body_text, DATE(p.post_date, 'unixepoch') as post_dt
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    JOIN posts p ON tr.post_id = p.post_id
    WHERE t.canonical_name = 'low dose naltrexone'
    AND tr.sentiment = 'positive' AND tr.signal_strength = 'strong'
    AND LENGTH(p.body_text) BETWEEN 80 AND 500
    ORDER BY RANDOM() LIMIT 5
''', conn)

ldn_neg = pd.read_sql('''
    SELECT p.body_text, DATE(p.post_date, 'unixepoch') as post_dt
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    JOIN posts p ON tr.post_id = p.post_id
    WHERE t.canonical_name = 'low dose naltrexone'
    AND tr.sentiment = 'negative' AND tr.signal_strength = 'strong'
    AND LENGTH(p.body_text) BETWEEN 80 AND 500
    ORDER BY RANDOM() LIMIT 3
''', conn)

mag_pos = pd.read_sql('''
    SELECT p.body_text, DATE(p.post_date, 'unixepoch') as post_dt
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    JOIN posts p ON tr.post_id = p.post_id
    WHERE t.canonical_name IN ('magnesium', 'magnesium glycinate')
    AND tr.sentiment = 'positive' AND tr.signal_strength = 'strong'
    AND LENGTH(p.body_text) BETWEEN 80 AND 500
    ORDER BY RANDOM() LIMIT 3
''', conn)

nic_pos = pd.read_sql('''
    SELECT p.body_text, DATE(p.post_date, 'unixepoch') as post_dt
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    JOIN posts p ON tr.post_id = p.post_id
    WHERE t.canonical_name = 'nicotine'
    AND tr.sentiment = 'positive' AND tr.signal_strength = 'strong'
    AND LENGTH(p.body_text) BETWEEN 80 AND 500
    ORDER BY RANDOM() LIMIT 3
''', conn)

ssri_neg = pd.read_sql('''
    SELECT p.body_text, DATE(p.post_date, 'unixepoch') as post_dt
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    JOIN posts p ON tr.post_id = p.post_id
    WHERE t.canonical_name IN ('ssri', 'fluvoxamine', 'sertraline', 'duloxetine')
    AND tr.sentiment = 'negative'
    AND LENGTH(p.body_text) BETWEEN 80 AND 500
    ORDER BY RANDOM() LIMIT 3
''', conn)

html_parts = ['<div style="padding: 10px;">']

def add_quote_section(html_parts, title, df, max_quotes=2):
    html_parts.append(f'<h4>{title}</h4>')
    for i, row in df.head(max_quotes).iterrows():
        q = clean_quote(row.body_text)
        html_parts.append(f'<blockquote style="border-left: 3px solid #3498db; padding-left: 12px; color: #555; margin: 8px 0;"><em>"{q}"</em><br/><span style="font-size:0.85em; color:#888;">-- r/covidlonghaulers user, {row.post_dt}</span></blockquote>')

add_quote_section(html_parts, "LDN: Strong positive reports from the community's most-discussed treatment", ldn_pos, 2)
add_quote_section(html_parts, "LDN: But not everyone benefits -- sensitivity and side effects are real", ldn_neg, 1)
add_quote_section(html_parts, "Magnesium: Consistent relief with minimal downsides", mag_pos, 1)
add_quote_section(html_parts, "Nicotine patches: Unconventional but generating real enthusiasm", nic_pos, 1)
add_quote_section(html_parts, "SSRIs: The treatment class patients push back on", ssri_neg, 1)

html_parts.append('</div>')
display(HTML('\n'.join(html_parts)))


## 8. Sensitivity Check

Do the main conclusions hold if we restrict to strong-signal reports only? This filters out mentions that the NLP pipeline tagged as "weak" signal -- passing references, secondhand reports, or ambiguous context.

In [ ]:
# ── Sensitivity: strong-signal only ──
raw_strong = pd.read_sql('''
    SELECT tr.user_id, t.canonical_name as drug,
           AVG(CASE tr.sentiment
               WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5
               WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_score
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
    WHERE tr.signal_strength = 'strong'
    GROUP BY tr.user_id, t.canonical_name
''', conn)

raw_strong = raw_strong[~raw_strong.drug.isin(GENERIC_TERMS | CAUSAL_EXCLUDE)].copy()
raw_strong['drug_merged'] = raw_strong.drug.map(lambda d: MERGE_MAP.get(d, d))
user_drug_strong = raw_strong.groupby(['user_id', 'drug_merged']).agg(avg_score=('avg_score', 'mean')).reset_index()
user_drug_strong['is_positive'] = (user_drug_strong.avg_score > 0).astype(int)

strong_stats = user_drug_strong.groupby('drug_merged').agg(
    n_users=('user_id', 'nunique'),
    n_positive=('is_positive', 'sum'),
).reset_index()
strong_stats['pos_rate_strong'] = strong_stats.n_positive / strong_stats.n_users

compare = top[['drug_merged', 'display_name', 'pos_rate', 'n_users']].merge(
    strong_stats[['drug_merged', 'pos_rate_strong', 'n_users']].rename(columns={'n_users': 'n_strong'}),
    on='drug_merged', how='left'
)
compare = compare.dropna(subset=['pos_rate_strong'])
compare['delta'] = compare.pos_rate_strong - compare.pos_rate
compare = compare.sort_values('pos_rate', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot([0.2, 1.0], [0.2, 1.0], 'k--', alpha=0.3, label='No change line')

for _, row in compare.iterrows():
    color = '#2ecc71' if row.delta >= 0 else '#e74c3c'
    ax.scatter(row.pos_rate, row.pos_rate_strong, s=max(row.n_strong * 3, 30),
              color=color, alpha=0.7, edgecolors='white', linewidth=0.5)

# Add labels with overlap check
texts_list = []
for _, row in compare.iterrows():
    t = ax.annotate(row.display_name, (row.pos_rate + 0.008, row.pos_rate_strong + 0.008), fontsize=8)
    texts_list.append(t)

try:
    renderer = fig.canvas.get_renderer()
    for i, t1 in enumerate(texts_list):
        bb1 = t1.get_window_extent(renderer)
        for t2 in texts_list[i+1:]:
            bb2 = t2.get_window_extent(renderer)
            if bb1.overlaps(bb2):
                pos = t2.get_position()
                t2.set_position((pos[0], pos[1] + 0.025))
except Exception:
    pass

ax.set_xlabel('Positive Rate (All Signals)', fontsize=11)
ax.set_ylabel('Positive Rate (Strong Signals Only)', fontsize=11)
ax.set_title('Sensitivity Check: Do Conclusions Change with Strong-Signal Reports Only?\n(Dot size = strong-signal users; green = rate improved, red = rate decreased)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')

fig.tight_layout()
plt.show()

corr = compare[['pos_rate', 'pos_rate_strong']].corr().iloc[0, 1]
robust_text = "robust" if corr > 0.85 else ("moderately robust" if corr > 0.7 else "fragile")
detail_text = ("restricting to strong signals does not materially change the treatment ranking." if corr > 0.85
               else "some treatments shift position when restricted to strong signals, suggesting the ranking is somewhat sensitive to signal quality.")
display(HTML(f'<p><b>Correlation between all-signal and strong-signal rankings:</b> r = {corr:.3f}. The main conclusions are {robust_text} -- {detail_text}</p>'))


## 9. Tiered Recommendations

Based on the statistical evidence above, treatments are classified into three tiers. **Strong** recommendations have n >= 30 users and p < 0.05 vs the 50% baseline. **Moderate** recommendations have n >= 15 or p < 0.10. **Preliminary** recommendations have fewer than 15 users -- interesting but requiring more evidence.

In [ ]:
# ── Tiered recommendations with visual summary ──
def assign_tier(row):
    bt = binomtest(int(row.n_positive), int(row.n_users), 0.5)
    p = bt.pvalue
    if row.n_users >= 30 and p < 0.05 and row.pos_rate > 0.5:
        return 'Strong'
    elif row.n_users >= 15 and (p < 0.10 or row.pos_rate > 0.6):
        return 'Moderate'
    else:
        return 'Preliminary'

top['tier'] = top.apply(assign_tier, axis=1)

for tier_name, color, border in [('Strong', '#27ae60', '#1e8449'), ('Moderate', '#f39c12', '#d68910'), ('Preliminary', '#95a5a6', '#717d7e')]:
    tier_df = top[top.tier == tier_name].sort_values('wilson_lb', ascending=False)
    if len(tier_df) == 0:
        continue

    fig, ax = plt.subplots(figsize=(10, max(3, len(tier_df) * 0.45 + 1)))

    y = range(len(tier_df))
    bars = ax.barh(y, tier_df.pos_rate * 100, height=0.6, color=color, alpha=0.8, edgecolor=border)

    xerr_lo = (tier_df.pos_rate - tier_df.ci_lo) * 100
    xerr_hi = (tier_df.ci_hi - tier_df.pos_rate) * 100
    ax.errorbar(tier_df.pos_rate * 100, y, xerr=[xerr_lo.values, xerr_hi.values],
                fmt='none', color='#333', capsize=3, linewidth=1.2)

    ax.set_yticks(y)
    ax.set_yticklabels([f"{r.display_name} (n={int(r.n_users)})" for _, r in tier_df.iterrows()], fontsize=10)
    ax.axvline(50, color='#e74c3c', linestyle=':', alpha=0.5, label='50% chance')
    ax.axvline(baseline_rate * 100, color='grey', linestyle='--', alpha=0.5, label=f'Baseline: {baseline_rate:.0%}')
    ax.set_xlabel('Positive Outcome Rate (%)')
    ax.set_title(f'{tier_name} Evidence Tier ({len(tier_df)} treatments)', fontsize=13, fontweight='bold', color=border)
    ax.set_xlim(15, 110)
    ax.legend(loc='lower right', fontsize=9)

    for i, (_, row) in enumerate(tier_df.iterrows()):
        ax.text(min(row.pos_rate * 100 + 2, 100), i, f"{row.pos_rate:.0%}", va='center', fontsize=9, fontweight='bold')

    fig.tight_layout()
    plt.show()


**Strong tier** treatments have large sample sizes and statistically significant positive outcomes. LDN stands out as the strongest overall recommendation -- high positive rate (74%) with the largest evidence base (n=183). Magnesium, vitamin D, electrolytes, and B12 are low-risk, accessible, and well-supported by this data.

**Moderate tier** treatments are promising but have wider confidence intervals. Nicotine patches, nattokinase, and GLP-1 agonists are generating community interest but need more evidence.

**Note on SSRIs and cromolyn sodium:** These appear in the lower tiers not because they are definitively ineffective, but because this community's experience with them is mixed-to-negative. A patient with a clear clinical indication (diagnosed depression, confirmed MCAS) should discuss these with their doctor regardless of community sentiment.

## 10. Conclusion

In [ ]:
# ── Conclusion summary ──
n_total = len(top)

display(HTML(f'''
<div style="background: #eaf5ea; padding: 18px; border-radius: 8px; border-left: 5px solid #27ae60;">
<h4 style="margin-top:0;">Bottom Line</h4>
<p>Out of {n_total} treatments with sufficient data, the Long COVID community on Reddit most consistently reports positive outcomes from <b>micronutrient supplementation</b> (magnesium, B vitamins, vitamin D, electrolytes, quercetin) and <b>low dose naltrexone (LDN)</b>. These are not fringe findings -- LDN alone has 183 user reports in one month, making it the most discussed treatment in the dataset.</p>

<p>A patient new to Long COVID, based on this data, should consider starting with <b>magnesium, electrolytes, and B vitamins</b> as low-risk, accessible, and well-supported first steps. <b>LDN</b> is the strongest single-agent recommendation for those who can access it (it requires a prescription and compounding pharmacy). <b>Nicotine patches</b> are an unconventional option generating real enthusiasm in this community, though they carry their own risks.</p>

<p><b>SSRIs should be approached with caution</b> for Long COVID specifically. While they may help comorbid depression or anxiety, this community reports them as no better than chance for Long COVID symptoms overall. <b>Cromolyn sodium</b>, despite its theoretical appeal for MCAS, has the worst outcome profile in this dataset.</p>

<p>The most surprising finding is the magnitude of the gap between simple, cheap micronutrients and expensive prescription medications. This likely reflects a combination of factors: micronutrient deficiencies are genuinely common in Long COVID and respond well to correction; prescription medications are prescribed for more severe cases (confounding by indication); and community reporting favors accessible interventions. Regardless of the cause, the pattern is consistent and large enough to be clinically interesting.</p>
</div>
'''))


## 11. Research Limitations

1. **Selection bias:** This data comes from r/covidlonghaulers on Reddit. Reddit users skew younger, more tech-savvy, and more engaged with self-advocacy than the general Long COVID population. Treatment preferences and outcomes may differ in populations not represented on Reddit.

2. **Reporting bias:** Users are more likely to post about treatments that produced a strong response (positive or negative) than treatments that did nothing. This inflates both tails and underrepresents null results. The 74% overall positive rate is a symptom of this bias.

3. **Survivorship bias:** Users who recovered fully may leave the community and stop posting. Users who are too ill to post are absent from the data. The "surviving" community members represent a middle band of severity -- not the best or worst outcomes.

4. **Recall bias:** Posts are written from memory. Users may misattribute improvements to the most recent treatment change, forget timing details, or conflate multiple treatments started simultaneously.

5. **Confounding:** Users taking LDN are likely also taking magnesium, B vitamins, and other supplements. We cannot isolate individual treatment effects from this observational data. The heatmap in Section 5 is hypothesis-generating, not causal.

6. **No control group:** There is no untreated comparison group. We compare treatments to each other and to a statistical baseline, but we cannot compare to "no treatment at all." Some positive outcomes may reflect natural disease trajectory, not treatment effect.

7. **Sentiment vs. efficacy:** NLP-extracted sentiment is a proxy for patient-reported outcomes, not a clinical measurement. A "positive" sentiment report could mean symptom reduction, hope about a new treatment, or simply a less negative experience than expected. We do not have biomarkers, functional assessments, or standardized outcome measures.

8. **Temporal snapshot:** One month of data (March 11 -- April 10, 2026) captures a snapshot, not a trend. Treatment popularity and outcomes may shift with new research, new products entering the market, or changes in community composition. The GLP-1 agonist discussion, for example, appears to be a recent trend that may or may not persist.

In [ ]:
display(HTML('<div style="font-size: 1.2em; font-weight: bold; font-style: italic; text-align: center; margin: 30px 0; padding: 20px; background: #fff3cd; border-radius: 8px;">"These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice."</div>'))
conn.close()
